In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import GroupShuffleSplit

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from sklearn.ensemble import RandomForestRegressor

from xgboost import XGBRegressor

pd.set_option("display.max_columns", None)

In [2]:
df = pd.read_csv("../data/processed/delivery_data_features.csv")

print(df.shape)

(142502, 41)


In [3]:
target = "segment_actual_time"

drop_cols = [
    "trip_uuid",
    "route_schedule_uuid",
    "segment_actual_time",
    "actual_time",
    "factor",
    "segment_factor",
    "trip_creation_time",
    "od_start_time",
    "od_end_time",
    "cutoff_timestamp",
    "source_name",
    "destination_name",
    "corridor",
    "trip_total_actual_time"
]

X = df.drop(columns=drop_cols)
y = df[target]

print(X.shape)
print(y.shape)

(142502, 27)
(142502,)


In [4]:
from sklearn.preprocessing import LabelEncoder

label_cols = [
    "data",
    "route_type",
    "source_center",
    "destination_center"
]

encoders = {}

for col in label_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))
    encoders[col] = le

print("Encoding complete.")

Encoding complete.


In [5]:
groups = df["trip_uuid"]

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

print(X_train.shape)
print(X_test.shape)

(114541, 27)
(27961, 27)


In [6]:
from xgboost import XGBRegressor

xgb = XGBRegressor(
    n_estimators=300,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

xgb.fit(X_train, y_train)

print("XGBoost training complete.")

XGBoost training complete.


In [7]:
y_pred_xgb = xgb.predict(X_test)

print("Predictions complete.")

Predictions complete.


In [8]:
mae = mean_absolute_error(y_test, y_pred_xgb)

mse = mean_squared_error(
    y_test,
    y_pred_xgb
)

rmse = mse ** 0.5

r2 = r2_score(
    y_test,
    y_pred_xgb
)

print("MAE :", round(mae, 2))
print("RMSE:", round(rmse, 2))
print("R²  :", round(r2, 4))

MAE : 10.6
RMSE: 24.13
R²  : 0.7184


In [9]:
feature_importance_xgb = pd.DataFrame({
    "Feature": X.columns,
    "Importance": xgb.feature_importances_
})

feature_importance_xgb = feature_importance_xgb.sort_values(
    "Importance",
    ascending=False
)

feature_importance_xgb.head(15)

,Feature,Importance
23,historical_delay_ratio,0.159088
10,segment_osrm_time,0.112662
5,is_cutoff,0.100700
11,segment_osrm_distance,0.091030
26,trip_avg_speed,0.068667
4,start_scan_to_end_scan,0.049376
24,trip_segments,0.039124
6,cutoff_factor,0.035252
9,osrm_distance,0.031495
22,historical_corridor_delay,0.029314


In [10]:
import joblib

joblib.dump(
    xgb,
    "../models/final_eta_model.pkl"
)

print("Model saved successfully.")

Model saved successfully.


In [11]:
joblib.dump(
    encoders,
    "../models/label_encoders.pkl"
)

print("Encoders saved successfully.")

Encoders saved successfully.


In [12]:
comparison = pd.DataFrame({
    "Model": [
        "Random Forest",
        "XGBoost"
    ],
    "MAE": [
        10.33,
        10.60
    ],
    "RMSE": [
        24.58,
        24.13
    ],
    "R2": [
        0.7078,
        0.7184
    ]
})

comparison

,Model,MAE,RMSE,R2
0,Random Forest,10.33,24.58,0.7078
1,XGBoost,10.60,24.13,0.7184


In [13]:
comparison.to_csv(
    "../reports/model_comparison.csv",
    index=False
)

print("Comparison table saved.")

Comparison table saved.


# Final Model Selection

Models Evaluated:

1. Random Forest
2. XGBoost

Results:

| Model | MAE | RMSE | R² |
|---------|---------|---------|---------|
| Random Forest | 10.33 | 24.58 | 0.7078 |
| XGBoost | 10.60 | 24.13 | 0.7184 |

Final Model Selected:
XGBoost

Reason:
- Highest R²
- Lowest RMSE
- Better overall generalization

Artifacts Saved:
- final_eta_model.pkl
- label_encoders.pkl

# Conclusion

The XGBoost model was selected as the final ETA prediction model.

Final Performance:

- MAE: 10.60 minutes
- RMSE: 24.13 minutes
- R²: 0.7184

Artifacts Generated:

- final_eta_model.pkl
- label_encoders.pkl
- model_comparison.csv

The model will be used for dashboard integration and ETA prediction.